# 02 — Analisis Event Registry

Seluruh angka pada notebook ini berasal dari **data operasi nyata**
PLTU Jeranjang Unit 1, Januari 2016 sampai Mei 2026. Inilah bagian
laporan yang boleh dikutip apa adanya.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from backend.app.core.config import get_settings
settings = get_settings()
print("Akar proyek:", settings.paths.root)
print("Mode deployment:", settings.deployment_mode)

In [ ]:
from backend.app.data.event_etl import load_registry

registry = load_registry(settings)
print(f"{len(registry)} event")
registry["event_type"].value_counts().to_frame("jumlah")

## Dampak per jenis event

In [ ]:
from backend.app.reports.event_analysis import impact_by_event_type

impact_by_event_type(registry)

## Temuan: tidak ada satu pun catatan blocking cold slag pipe

README §5 menempatkan blocking slag pipe sebagai target utama, lengkap
dengan empat target probabilitas per pipa. Sepanjang sepuluh tahun,
jurnal gangguan tidak pernah mencatatnya.

In [ ]:
blocking_types = settings.event_taxonomy["blocking_event_types"]
counts = registry["event_type"].value_counts()
for event_type in blocking_types:
    print(f"{event_type:32}: {counts.get(event_type, 0):>3}")

## Tren tahunan

In [ ]:
from backend.app.reports.event_analysis import yearly_trend

trend = yearly_trend(registry, blocking_types)
trend.assign(total=trend.sum(axis=1))

## Kekambuhan

Indeks dispersi di atas 1,0 berarti event menggerombol. Proses acak tanpa
memori bernilai sekitar 1,0.

In [ ]:
from backend.app.reports.event_analysis import recurrence_stats

for label, types in [
    ("Semua blocking", blocking_types),
    ("Aglomerasi", ["furnace_agglomeration"]),
    ("Blocking feeder", ["coal_feeder_blocking"]),
]:
    stats = recurrence_stats(registry, types)
    if stats.get("insufficient_data"):
        print(f"{label:18}: data tidak cukup")
        continue
    print(
        f"{label:18}: n={stats['count']:>3}  "
        f"median {stats['gap_days_median']:>7.1f} hari  "
        f"dispersi {stats['dispersion_index']:.2f}  "
        f"{'MENGGEROMBOL' if stats['clustered'] else 'menyebar'}"
    )

## Musiman

Hipotesis awal: blocking feeder menumpuk pada musim hujan karena batubara
lembab. Data menolaknya.

In [ ]:
from backend.app.reports.event_analysis import monthly_profile

monthly = monthly_profile(registry, ["coal_feeder_blocking", "furnace_agglomeration"])
wet = ["Nov", "Des", "Jan", "Feb", "Mar", "Apr"]
dry = ["Mei", "Jun", "Jul", "Agu", "Sep", "Okt"]
print("Musim hujan (Nov-Apr) vs kemarau (Mei-Okt)")
for column in monthly.columns:
    print(f"  {column:24}: {monthly.loc[wet, column].sum():>3} vs {monthly.loc[dry, column].sum():>3}")
monthly

## Grafik

In [ ]:
from backend.app.reports.event_analysis import run_analysis

summary = run_analysis(settings)
for name, path in summary["figures"].items():
    print(f"{name:20}: {path}")

In [ ]:
from IPython.display import Image, display

for path in summary["figures"].values():
    display(Image(filename=path))

## Event yang perlu tinjauan engineer

In [ ]:
review = registry.loc[registry["needs_review"].fillna(False).astype(bool)]
print(f"{len(review)} event menunggu verifikasi")
review[["event_id", "start_time", "equipment_raw", "initial_symptom", "event_type"]].head(20)